In [1]:
# ── Imports ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
# ──  Load Data (Kaggle paths) ─────────────────────────────────────────
df_train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
df_test  = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

In [3]:
from sklearn.preprocessing import StandardScaler

# 1. Row-by-row feature engineering (No whole-dataset statistics involved)
def engineer_features_base(df):
    df = df.copy()
    
    # Drop string/identifier columns but keep PassengerId
    df.drop(['Cabin', 'Name', 'Ticket'], axis=1, inplace=True, errors='ignore')

    # Embarked
    df['Embarked'].fillna('S', inplace=True)
    df.replace({"Embarked": {"S": 0, "C": 1, "Q": 2}}, inplace=True)

    # Sex
    df.replace({"Sex": {"male": 0, "female": 1}}, inplace=True)

    # Family size
    df['SumPeople'] = df['SibSp'].astype(int) + df['Parch'].astype(int) + 1
    
    return df

# Apply base engineering
df_train = engineer_features_base(df_train)
df_test  = engineer_features_base(df_test)

# 2. Fill NaNs using TRAIN statistics only
train_age_median = df_train['Age'].median()
train_fare_median = df_train['Fare'].median()

male_mean   = 30.7266445916114
female_mean = 27.9157081226057

for df in [df_train, df_test]:
    # Fill Age using hardcoded sex means, then the Train median
    df.loc[(df['Sex'] == 0) & (df['Age'].isnull()), 'Age'] = male_mean
    df.loc[(df['Sex'] == 1) & (df['Age'].isnull()), 'Age'] = female_mean
    df['Age'].fillna(train_age_median, inplace=True)
    
    # Fill Fare using the Train median
    df['Fare'].fillna(train_fare_median, inplace=True)

    # Age group (uses fixed bins, safe to loop)
    age        = [0, 5, 15, 25, 30, 35, 45, 50, 200]
    age_label  = ['0-5','5-15','15-25','25-30','30-35','35-40','45-50','>50']
    df['age_group']      = pd.cut(df['Age'], age, labels=age_label)
    df['age_group_code'] = df['age_group'].cat.codes

    # Fare group (uses fixed bins, safe to loop)
    price       = [0, 10, 30, 35, 80, 1000]
    price_label = ['0-10','10-30','30-35','35-80','>80']
    df['price_group']      = pd.cut(df['Fare'], price, labels=price_label)
    df['price_group_code'] = df['price_group'].cat.codes
    
    df.drop(['age_group', 'price_group'], axis=1, inplace=True, errors='ignore')

# 3. Scale features using TRAIN statistics only
scaler = StandardScaler()
numeric_cols = df_train.select_dtypes(include=[np.number]).columns
cols_to_scale = [col for col in numeric_cols if col not in ['Survived', 'PassengerId']]

# FIT on Train, TRANSFORM on Train
df_train[cols_to_scale] = scaler.fit_transform(df_train[cols_to_scale])

# TRANSFORM ONLY on Test (Crucial: Do not call fit_transform here!)
df_test[cols_to_scale]  = scaler.transform(df_test[cols_to_scale])

In [4]:
df_train.isnull().sum()

PassengerId         0
Survived            0
Pclass              0
Sex                 0
Age                 0
SibSp               0
Parch               0
Fare                0
Embarked            0
SumPeople           0
age_group_code      0
price_group_code    0
dtype: int64

In [5]:
df_test.isnull().sum()

PassengerId         0
Pclass              0
Sex                 0
Age                 0
SibSp               0
Parch               0
Fare                0
Embarked            0
SumPeople           0
age_group_code      0
price_group_code    0
dtype: int64

In [6]:
# ──  Prepare arrays ───────────────────────────────────────────────────
FEATURES = ['Pclass', 'Sex', 'age_group_code', 'price_group_code',
            'SumPeople', 'SibSp', 'Parch', 'Embarked']

X_train = df_train[FEATURES].values.astype(float)
y_train = df_train['Survived'].values.astype(float)
X_test  = df_test[FEATURES].values.astype(float)

# Normalise to [0,1] using train statistics
X_min = X_train.min(axis=0)
X_max = X_train.max(axis=0)
X_train = (X_train - X_min) / (X_max - X_min + 1e-8)
X_test  = (X_test  - X_min) / (X_max - X_min + 1e-8)

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")

X_train: (891, 8)  |  X_test: (418, 8)


In [7]:
import numpy as np
from tqdm import tqdm

# ──  Forward-Forward Network (Hinton, 2022) ───────────────────────────

class FFALayer:
    """A single layer trained via Forward-Forward (No Backprop)"""
    def __init__(self, in_feat, out_feat, lr=0.05, threshold=2.0):
        # Xavier-ish initialization
        self.W = np.random.randn(in_feat, out_feat) * np.sqrt(2.0 / in_feat)
        self.b = np.zeros((1, out_feat))
        self.lr = lr
        self.threshold = threshold

    def forward(self, x):
        # Basic linear + ReLU
        return np.maximum(0, x @ self.W + self.b)

    def normalize(self, h):
        # Normalize activities so large vectors don't blow up subsequent layers
        return h / (np.linalg.norm(h, axis=1, keepdims=True) + 1e-8)

    def train_step(self, x_pos, x_neg):
        N = x_pos.shape[0]

        # Forward passes
        h_pos = self.forward(x_pos)
        h_neg = self.forward(x_neg)

        # "Goodness" is just the sum of squared activations
        g_pos = np.sum(h_pos**2, axis=1, keepdims=True)
        g_neg = np.sum(h_neg**2, axis=1, keepdims=True)

        # Safe sigmoid calculation to prevent overflow
        def sigmoid(x):
            return 1.0 / (1.0 + np.exp(-np.clip(x, -100, 100)))

        # Probabilities based on Goodness vs Threshold
        p_pos = sigmoid(g_pos - self.threshold)
        p_neg = sigmoid(g_neg - self.threshold)

        # Gradients of the Loss w.r.t Goodness
        # Loss = -log(p_pos) - log(1 - p_neg)
        d_g_pos = p_pos - 1.0
        d_g_neg = p_neg

        # Gradients of Goodness w.r.t Weights (derived manually, pure math!)
        grad_W = (2 * x_pos.T @ (d_g_pos * h_pos) + 2 * x_neg.T @ (d_g_neg * h_neg)) / N
        grad_b = (2 * np.sum(d_g_pos * h_pos, axis=0, keepdims=True) + 2 * np.sum(d_g_neg * h_neg, axis=0, keepdims=True)) / N

        # Local weight update (No backward pass!)
        self.W -= self.lr * grad_W
        self.b -= self.lr * grad_b

        # Return normalized vectors to feed the next layer
        return self.normalize(h_pos), self.normalize(h_neg)


class ForwardForwardNetwork:
    """Trains layers greedily. Replaces labels with one-hot vectors overlaying the input."""
    def __init__(self, layer_sizes, lr=0.03, threshold=2.0, epochs_per_layer=100):
        self.layers = []
        self.epochs = epochs_per_layer
        for i in range(len(layer_sizes) - 1):
            self.layers.append(FFALayer(layer_sizes[i], layer_sizes[i+1], lr, threshold))

    def fit(self, X, y, verbose=True):
        # 1. Overlay labels on input data
        # We append a 2-dim one-hot vector to the 8-dim features (Total input dim = 10)
        y_one_hot = np.zeros((X.shape[0], 2))
        y_one_hot[np.arange(X.shape[0]), y.astype(int)] = 1.0
        
        y_neg_one_hot = 1.0 - y_one_hot # The exact opposite / fake label

        # Initialize the inputs for the FIRST layer
        h_p = np.hstack([X, y_one_hot])
        h_n = np.hstack([X, y_neg_one_hot])

        # 2. Train greedily, layer by layer
        for i, layer in enumerate(self.layers):
            loop = tqdm(range(self.epochs), desc=f"Training Layer {i+1}", disable=not verbose)
            
            # Train the current layer for N epochs on its STATIC inputs
            for _ in loop:
                layer.train_step(h_p, h_n)
            
            # AFTER the layer is fully trained, push the data through it 
            # to generate the inputs for the NEXT layer!
            h_p = layer.normalize(layer.forward(h_p))
            h_n = layer.normalize(layer.forward(h_n))

    def _calc_goodness(self, x, label):
        """Measures how much the network 'likes' a given label for the input."""
        y_test = np.zeros((x.shape[0], 2))
        y_test[:, label] = 1.0
        h = np.hstack([x, y_test])
        
        total_g = np.zeros((x.shape[0],))
        for layer in self.layers:
            h_unnorm = layer.forward(h)
            total_g += np.sum(h_unnorm**2, axis=1) # Sum goodness across all layers
            h = layer.normalize(h_unnorm)
        return total_g

    def predict(self, X):
        # Does the network prefer label 0 or label 1?
        g_0 = self._calc_goodness(X, 0)
        g_1 = self._calc_goodness(X, 1)
        return (g_1 > g_0).astype(int)

In [8]:
"""
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ── CELL 6: Optuna Hyperparameter Search for FFA ─────────────────────────────

# Split the data so Optuna can validate on unseen samples
X_trn, X_val, y_trn, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

def objective(trial):
    # 1. Define the FFA hyperparameter search space
    lr        = trial.suggest_float("lr", 1e-3, 2e-1, log=True)
    threshold = trial.suggest_float("threshold", 1.0, 10.0)
    epochs    = trial.suggest_int("epochs_per_layer", 50, 400)
    
    # Let Optuna evolve the brain architecture dynamically!
    hidden_1  = trial.suggest_int("hidden_1", 32, 128, step=16)
    hidden_2  = trial.suggest_int("hidden_2", 16, 64, step=16)
    
    n_feat = X_trn.shape[1]
    
    # 2. Instantiate the FFA network
    # Input must be n_feat + 2 for the one-hot label overlay
    model = ForwardForwardNetwork(
        layer_sizes=[n_feat + 2, hidden_1, hidden_2],
        lr=lr,
        threshold=threshold,
        epochs_per_layer=epochs
    )
    
    # 3. Train the model (silently, to avoid tqdm spam)
    model.fit(X_trn, y_trn, verbose=False)
    
    # 4. Evaluate and return the validation accuracy
    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)
    
    return acc

print(f"Starting Optuna optimization for Forward-Forward Network...\n")

# Maximize validation accuracy
study = optuna.create_study(direction="maximize")

# Run 30 trials with a progress bar for the overall study
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nBest Validation Accuracy: {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

"""

'\nimport optuna\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import accuracy_score\n\n# ── CELL 6: Optuna Hyperparameter Search for FFA ─────────────────────────────\n\n# Split the data so Optuna can validate on unseen samples\nX_trn, X_val, y_trn, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)\n\ndef objective(trial):\n    # 1. Define the FFA hyperparameter search space\n    lr        = trial.suggest_float("lr", 1e-3, 2e-1, log=True)\n    threshold = trial.suggest_float("threshold", 1.0, 10.0)\n    epochs    = trial.suggest_int("epochs_per_layer", 50, 400)\n    \n    # Let Optuna evolve the brain architecture dynamically!\n    hidden_1  = trial.suggest_int("hidden_1", 32, 128, step=16)\n    hidden_2  = trial.suggest_int("hidden_2", 16, 64, step=16)\n    \n    n_feat = X_trn.shape[1]\n    \n    # 2. Instantiate the FFA network\n    # Input must be n_feat + 2 for the one-hot label overlay\n    model = ForwardForwardNetwork(\n  

In [9]:
# ── Train Forward-Forward ────────────────────────────────────────────

np.random.seed(42)
n_feat = X_train.shape[1] 

# Input = 8 features + 2 for the one-hot label overlay = 10 input neurons.
# We don't need a single output node like normal ML. We just use hidden blocks.
ffa_model = ForwardForwardNetwork(
    layer_sizes=[n_feat + 2, 64, 32], 
    lr=0.11472652365842113, 
    threshold=5.977718087903543, 
    epochs_per_layer=5000
)

print(f"FFA Architecture: {n_feat}+2 → 64 → 32 | NO backpropagation\n")
ffa_model.fit(X_train, y_train)

FFA Architecture: 8+2 → 64 → 32 | NO backpropagation



Training Layer 2: 100%|██████████| 5000/5000 [00:07<00:00, 649.60it/s]


In [10]:
# ── Evaluate ─────────────────────────────────────────────────────────
train_acc = np.mean(ffa_model.predict(X_train) == y_train)
print(f"\nFinal Train Accuracy (FFA): {train_acc:.4f}")


Final Train Accuracy (FFA): 0.8339


In [11]:
# ── Submission ───────────────────────────────────────────────────────
test_preds = ffa_model.predict(X_test)
submission = pd.DataFrame({
    "PassengerId": df_test["PassengerId"],
    "Survived":    test_preds
})
submission.to_csv("submission.csv", index=False)
print(f"\nSaved: submission.csv")
print(f"Predicted survivors: {test_preds.sum()} / {len(test_preds)}")
print(submission.head(20))


Saved: submission.csv
Predicted survivors: 138 / 418
    PassengerId  Survived
0           892         0
1           893         0
2           894         0
3           895         0
4           896         0
5           897         0
6           898         1
7           899         0
8           900         1
9           901         0
10          902         0
11          903         0
12          904         1
13          905         0
14          906         1
15          907         1
16          908         0
17          909         0
18          910         0
19          911         1


In [12]:
import pandas as pd

# ── Offline Oracle Evaluation ────────────────────────────────────────

# Load the local ground truth dataset you created
ground_truth_path = '/kaggle/input/datasets/chakrabhuanavdeva/groundtruth-titanic/submission.csv'
ground_truth = pd.read_csv(ground_truth_path)

# Merge predictions and ground truth on PassengerId to ensure perfect alignment
evaluation_df = submission.merge(ground_truth, on='PassengerId', suffixes=('_pred', '_true'))

# Calculate accuracy metrics
correct_predictions = (evaluation_df['Survived_pred'] == evaluation_df['Survived_true']).sum()
total_passengers = len(evaluation_df)
test_accuracy = correct_predictions / total_passengers

# Display the autopsy results
print("┌─────────────────────────────────────────────────┐")
print("│             OFFLINE ORACLE GRADING              │")
print("├─────────────────────────────────────────────────┤")
print(f"│ Total Evaluated:        {total_passengers:<24}│")
print(f"│ Correct Predictions:    {correct_predictions:<24}│")
print(f"│ Ground Truth Accuracy:  {test_accuracy:.4%}               │")
print("└─────────────────────────────────────────────────┘")

# Optional: See where the model specifically failed
errors = evaluation_df[evaluation_df['Survived_pred'] != evaluation_df['Survived_true']]
print(f"\nTotal misclassifications: {len(errors)}")

┌─────────────────────────────────────────────────┐
│             OFFLINE ORACLE GRADING              │
├─────────────────────────────────────────────────┤
│ Total Evaluated:        418                     │
│ Correct Predictions:    328                     │
│ Ground Truth Accuracy:  78.4689%               │
└─────────────────────────────────────────────────┘

Total misclassifications: 90
